In [ ]:
import os
from typing import Annotated, Sequence
from langchain_groq import ChatGroq
#from langchain_tavily import TavilySearchResults
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_tavily import TavilySearch
from IPython.display import Image,display

# Set your API Keys
os.environ["GROQ_API_KEY"] = "your_groq_api_key"
os.environ["TAVILY_API_KEY"] = "your_tavily_api_key"


In [31]:
# 2. Define the Search Tool
search_tool = TavilySearchResults(k=3)
tools = [search_tool]

# 3. Initialize the Groq Model and bind tools
# llama-3.3-70b is highly reliable for tool-calling logic
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).bind_tools(tools)

In [32]:
# 4. Define the Nodes
def call_model(state: MessagesState):
    """The 'Brain' node: decides if it needs to search or just answer."""
    response = model.invoke(state["messages"])
    return {"messages": [response]}

# Prebuilt ToolNode handles the actual execution of Tavily
tool_node = ToolNode(tools)

In [33]:
# 5. Define Routing Logic
def should_continue(state: MessagesState):
    """Decides if we loop back for more info or finish."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "action"
    return END

In [34]:
# 6. Build the Graph
workflow = StateGraph(MessagesState)

workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("action", "agent") # The Loop: observe search results & re-think
workflow.add_edge("action", END)

agent = workflow.compile()


In [35]:
# 7. Run the Search Agent
query = "What is the current temperature in Noida, Is there any chance of rain today?"
print(f"--- Researching: {query} ---")

for chunk in agent.stream({"messages": [HumanMessage(content=query)]}):
    for node, output in chunk.items():
        print(f"\n[Node: {node}]")
        if node == "agent":
            # Show the AI's thought process or final answer
            print(output["messages"][0].content or "Thinking... (Calling Search)")

--- Researching: What is the current temperature in Noida, Is there any chance of rain today? ---

[Node: agent]
Thinking... (Calling Search)

[Node: action]

[Node: agent]
The current temperature in Noida is 101°F (38°C) and it is expected to be hazy and very warm today. There is a chance of rain today, with a high of 101°F (38°C) and a low of 81°F (27°C). The weather forecast for the next 10 days shows a mix of hazy and sunny conditions, with temperatures ranging from 101°F (38°C) to 104°F (40°C). It is recommended to check the weather forecast regularly for updates.
